In [1]:
from guardrails import Guard
from langchain_chroma import Chroma
from pathlib import Path
from dotenv import load_dotenv
from config.parameter_config import params_config
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from guardrails_ai.provenance_embeddings import ProvenanceEmbeddings
from guardrails_ai.reading_time import ReadingTime
from guardrails_ai.relevancy_evaluator import RelevancyEvaluator
import numpy as np

In [2]:
load_dotenv()

True

In [3]:
app_params = params_config.rag_app

In [4]:
REPO_ROOT = Path.cwd().parent
VECTOR_STORE_DIR = REPO_ROOT / "saved-embeddings"

In [5]:
embedder = OpenAIEmbeddings(model=app_params.embedding_model,
                            dimensions=app_params.embedding_dimensions) # 1536

In [6]:
chunk_size = app_params.chunk_size
chunk_overlap = app_params.chunk_overlap

In [7]:
def load_knowledge_base():
    # vector store
    vs = Chroma(collection_name=app_params.collection_name,
            embedding_function=embedder,
            persist_directory=VECTOR_STORE_DIR.as_posix())


    return vs

In [8]:
if VECTOR_STORE_DIR.exists():
    vs = load_knowledge_base()

In [9]:
def get_retriever():
    # create the retriever
    retriever = vs.as_retriever(search_type=app_params.search_type,
                                search_kwargs={"k":app_params.k})
    

    if app_params.contextual_compression:
        # compressor
        compression_llm = ChatOpenAI(model=app_params.compression_llm)
        compressor = LLMChainExtractor.from_llm(compression_llm)
        
        # compression retriever
        compression_retriever = ContextualCompressionRetriever(
            base_compressor=compressor,
            base_retriever=retriever
        )
        return compression_retriever
    
    return retriever

retriever = get_retriever()

In [10]:
def embed_func(texts):
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

    if isinstance(texts, str):
        return np.array(embedding_model.embed_query(texts))

    return np.array(embedding_model.embed_documents(texts))

In [11]:
# generate the guardrails

# reading_time = ReadingTime(
#     reading_time=3,
#     on_fail="noop"
# )

relevancy_eval = RelevancyEvaluator(
    llm_callable="gpt-5-mini",
    on_fail="refrain"
)

hallucination_check = ProvenanceEmbeddings(
    threshold=0.7,
    validation_method="sentence",
    on_fail="refrain",
)

In [12]:
# create the guard

guard = Guard().use(
    # reading_time,
    relevancy_eval,
    hallucination_check
)

In [13]:
query = "What is the diffence between online vs offline evals"

In [14]:
retriever_output = retriever.invoke(query)

context = "\n\n".join([doc.page_content for doc in retriever_output])

In [15]:
llm = ChatOpenAI(model=app_params.llm)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant. Answer the user query
                  based on the given context only. If you do not know the answer
                  say I don't know. Do not add any preamble to the response"""),
    ("human", "context: {context}\n\nquery: {query}")
])

rag_chain = prompt | llm | StrOutputParser()
response = rag_chain.invoke({"context": context, "query": query})

In [16]:
print(response)

- Definition / where run
  - Online eval: runs in production after deployment as real users interact with the system.
  - Offline eval: runs on held-out / small datasets (outside production).

- Ground truth / answers
  - Online: works without an answer / without a golden dataset.
  - Offline: uses golden answers so you can compute exact metrics.

- Purpose / typical uses
  - Online: drift detection, measuring real-world chatbot performance (cost, speed), version comparison and CI in production.
  - Offline: fast experiments, repeatable comparisons during development.

- Cost & speed
  - Offline: fast, cheap and repeatable (running code/LLM on small data).
  - Online: more costly/slower (you may sample conversations — e.g. monitor 1,000 randomly — to reduce cost).

- Relationship
  - They are complementary, not rivals; both are used together.


In [17]:
context = """Offline evaluation happens before deployment, using a fixed golden dataset that you
create ahead of time. It's fast, cheap, and repeatable. Online evaluation happens
after deployment, on live production traffic, without a golden dataset."""


query = "What is the difference between offline and online evals?"

response = """MLflow was created by Databricks in 2018 and its enterprise tier costs $499/month,
which includes unlimited experiment tracking and AI copilot features."""

In [18]:
validation_metadata = {
    "sources": [context],
    "original_prompt": query,
    "embed_function": embed_func,
    "chunk_size": 2,
    "chunk_overlap": 1
}
try:
    outcome = guard.validate(response, metadata=validation_metadata)
    if outcome.validated_output is None:
        print("Please ask again. No relevant response generated")
    if outcome.validation_passed:
        print(f"Response: {outcome.validated_output}")
except Exception as e:
    print(e) 

c:\Users\asus\Desktop\LLMOPS Tutorial\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


Please ask again. No relevant response generated
